In [4]:
from langgraph.graph import StateGraph,START,END
from langchain_groq import ChatGroq
from typing import TypedDict,Literal,Annotated
from langchain_core.messages import SystemMessage,HumanMessage
from dotenv import load_dotenv
from pydantic import BaseModel, Field


In [5]:
generatormodel = ChatGroq(
    model="llama-3.1-8b-instant"  # free model
    
)
evalmodel = ChatGroq(
    model="llama-3.1-8b-instant"  # free model
    
)
optimizermodel = ChatGroq(
    model="llama-3.1-8b-instant"  # free model
    
)

In [21]:

class TweetEvaluation(BaseModel):
    evaluation: Literal["approved", "needs improvement"] = Field(..., description="Final evaluation result.")
    feedback: str = Field(..., description="feedback for the tweet.")

In [7]:
structured_evaluator_llm = evalmodel.with_structured_output(TweetEvaluation)


In [22]:
class TweetState(TypedDict):
    topic:str
    tweet:str
    eval:Literal['approved','needs improvement']
    feedback:str
    iteration:int
    max_itn:int

In [9]:
def generate(state:TweetState)->TweetState:
     #prompt
     messages = [
        SystemMessage(content="You are a funny and clever Twitter/X influencer."),
        HumanMessage(content=f"""
Write a short, original, and hilarious tweet on the topic: "{state['topic']}".

Rules:
- Do NOT use question-answer format.
- Max 280 characters.
- Use observational humor, irony, sarcasm, or cultural references.
- Think in meme logic, punchlines, or relatable takes.
- Use simple, day to day english

""")
    ]
     res=generatormodel.invoke(messages)
     return {"tweet": res.content}

In [23]:
def evaluate(state:TweetState)->TweetState:
    #prompt
    messages = [
    SystemMessage(content="You are a ruthless, no-laugh-given Twitter critic. You evaluate tweets based on humor, originality, virality, and tweet format."),
    HumanMessage(content=f"""
Evaluate the following tweet:

Tweet: "{state['tweet']}"

Use the criteria below to evaluate the tweet:

1. Originality – Is this fresh, or have you seen it a hundred times before?  
2. Humor – Did it genuinely make you smile, laugh, or chuckle?  
3. Punchiness – Is it short, sharp, and scroll-stopping?  
4. Virality Potential – Would people retweet or share it?  
5. Format – Is it a well-formed tweet (not a setup-punchline joke, not a Q&A joke, and under 280 characters)?

Auto-reject if:
- It's written in question-answer format (e.g., "Why did..." or "What happens when...")
- It exceeds 280 characters
- It reads like a traditional setup-punchline joke
- Dont end with generic, throwaway, or deflating lines that weaken the humor (e.g., “Masterpieces of the auntie-uncle universe” or vague summaries)

### Respond ONLY in structured format:
- evaluation: "approved" or "needs_improvement"  
- feedback: One paragraph explaining the strengths and weaknesses 
""")
]
    result = structured_evaluator_llm.invoke(messages)
    return {"eval": result.evaluation, "feedback": result.feedback}


In [11]:
def optimize(state:TweetState)->TweetState:
    #prompt
    messages = [
        SystemMessage(content="You punch up tweets for virality and humor based on given feedback."),
        HumanMessage(content=f"""
Improve the tweet based on this feedback:
"{state['feedback']}"

Topic: "{state['topic']}"
Original Tweet:
{state['tweet']}

Re-write it as a short, viral-worthy tweet. Avoid Q&A style and stay under 280 characters.
""")
    ]
    result = optimizermodel.invoke(messages).content
    iteration = state['iteration'] + 1
    return {"tweet": result, "iteration": iteration}

In [ ]:
def route_eval(state: TweetState):
    if state['eval']=='approved' or state['iteration']<=state['max_itn']:
        return 'approved'
    else:
        return 'needs improvement'

In [25]:
graph=StateGraph(TweetState)
graph.add_node('generate',generate)
graph.add_node('evaluate',evaluate)
graph.add_node('optimize',optimize)
graph.add_edge(START,'generate')
graph.add_edge('generate','evaluate')
graph.add_conditional_edges('evaluate',route_eval,{'approved':END,'needs improvement':'optimize'})
graph.add_edge('optimize','evaluate')

workflow=graph.compile()

In [ ]:
initial_state={
    "topic": "Indian Railways",
    "iteration":1,
    "max_itn":5
}
final_state=workflow.invoke(initial_state)
print(final_state)

{'topic': 'Indian Railways', 'tweet': '"Just spent 12 hours on Indian Railways and I\'ve finally mastered the art of meditating while getting squished by an elderly aunt #IndianRailways #SardarToSardar"', 'eval': 'approved', 'feedback': "This tweet is an excellent example of humor, originality, and virality potential. The writer's ability to poke fun at a relatable experience, Indian Railways, while cleverly incorporating a cultural reference (the 'Sardar' stereotype) makes it stand out. However, the punchiness could be improved by making the sentence structure more concise, but overall, it's a well-formed tweet that effectively uses humor to engage the reader.", 'iteration': 0, 'max_itn': 5}
